# CineData Analytics — Bronze → Silver
Limpeza, tipagem, tradução de colunas para português e regras de negócio.
**A Bronze não é alterada.** Como a Bronze usa *append*, cada tabela Silver parte da **ingestão mais recente** de cada chave.

In [ ]:
from pyspark.sql import functions as F, Window
from datetime import date

spark.sql("CREATE DATABASE IF NOT EXISTS silver")

def salva_silver(df, tabela):
    """Silver é derivada e reprocessável: overwrite garante idempotência do job."""
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(tabela))
    print(f"{tabela}: {spark.table(tabela).count()} linhas")

def mais_recente(df, chaves, cols_valor=None):
    """Deduplicação: mantém a versão mais recente (ingestion_datetime) por chave.
       Os duplicados da origem chegam na MESMA carga (mesmo ingestion_datetime), então o empate é resolvido por:
       1) registro mais completo (mais colunas de negócio preenchidas) e 2) hash do conteúdo (determinístico)."""
    cols_valor = cols_valor or [c for c in df.columns if c not in chaves + ["ingestion_datetime"]]
    completude = sum(F.when(F.col(c).isNotNull(), 1).otherwise(0) for c in cols_valor)
    w = Window.partitionBy(*chaves).orderBy(F.col("ingestion_datetime").desc(), completude.desc(),
                                            F.sha2(F.concat_ws("|", *[F.col(c).cast("string") for c in cols_valor]), 256))
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

# ---------- Helpers de limpeza (todos "à prova de erro": nunca quebram o pipeline) ----------
NULOS = ["", "unknown", "não informado", "nao informado", "n/a", "n/d", "na", "null", "none", "nan", "nenhum", "-", "--", "undefined", "[]"]
NUM = r"^[+-]?(\d+\.?\d*|\.\d+)([eE][+-]?\d+)?$"

def limpa_nulos(c):
    """Trim + converte textos que representam ausência de dado em NULL."""
    c = F.trim(F.col(c) if isinstance(c, str) else c)
    return F.when(F.lower(c).isin(NULOS), F.lit(None)).otherwise(c)

def to_double(c):
    """Cast seguro: texto incompatível vira NULL (regex valida antes do cast)."""
    c = limpa_nulos(c)
    return F.when(c.rlike(NUM), c.cast("double"))

def to_int(c):
    """Cast seguro para inteiro (remove milhar ',' e aceita '123.0'); negativos/fracionários/overflow viram NULL."""
    c = F.regexp_replace(limpa_nulos(c), r"(?<=\d),(?=\d{3}(\D|$))", "")
    d = F.when(c.rlike(NUM), c.cast("double"))
    return F.when((d >= 0) & (d == F.floor(d)) & (d <= 2147483647), d.cast("int"))

def limpa_decimal(c):
    """Popularidade com pontuação inconsistente: descobre qual separador é o decimal.
       '1.234,56' -> 1234.56 | '1,234.56' -> 1234.56 | '12,5' -> 12.5 | '1.234.567' -> 1234567 | '8.5' -> 8.5"""
    raw = limpa_nulos(c)
    # Column shift: textos soltos (sinopse, nomes) NÃO podem virar número ao "limpar" letras -> só segue se já for numérico
    s = F.when(raw.rlike(r"^-?[0-9][0-9.,]*$"), raw)
    n_pt = F.length(s) - F.length(F.regexp_replace(s, r"\.", ""))
    n_vg = F.length(s) - F.length(F.regexp_replace(s, ",", ""))
    s = (F.when((n_pt > 0) & (n_vg > 0),
                F.when(s.rlike(r",[^.]*$"), F.regexp_replace(F.regexp_replace(s, r"\.", ""), ",", "."))  # vírgula decimal
                 .otherwise(F.regexp_replace(s, ",", "")))                                                # ponto decimal
          .when(n_vg > 1, F.regexp_replace(s, ",", ""))
          .when(n_vg == 1, F.regexp_replace(s, ",", "."))
          .when(n_pt > 1, F.regexp_replace(s, r"\.(?=.*\.)", ""))   # mantém só o último ponto
          .otherwise(s))
    return F.when(s.rlike(r"^-?\d+(\.\d+)?$"), s.cast("double"))

def limpa_moeda(c):
    """Orçamento/receita: remove símbolos ($, US$, USD), milhar, centavos e textos; <= 0 vira NULL.
       Todo cast fica DENTRO de um when() com regex: seguro mesmo com ANSI ligado (padrão no Databricks serverless)."""
    raw = limpa_nulos(c)
    # Sufixos de escala vistos na origem: "34.0M" = 34 milhões ("K" = mil, "B" = bilhão)
    padrao_abrev = r"^[^\d]*(\d+(?:\.\d+)?)\s*([KkMmBb])$"
    sufixo = F.upper(F.regexp_extract(raw, padrao_abrev, 2))
    base = F.regexp_extract(raw, padrao_abrev, 1)
    mult = F.when(sufixo == "K", 1e3).when(sufixo == "M", 1e6).when(sufixo == "B", 1e9)
    abreviado = F.when(sufixo != "", base.cast("double") * mult)
    cientifica = F.when(raw.rlike(r"^\d+(\.\d+)?[eE]\+?\d+$"), raw.cast("double"))
    s = F.regexp_replace(raw, r"[^0-9.,\-]", "")
    s = F.regexp_replace(s, r"[.,]\d{1,2}$", "")   # remove casas decimais (".00", ",5")
    s = F.regexp_replace(s, r"[.,]", "")           # remove separador de milhar
    formato_ok = raw.rlike(r"(?i)^(us\$|usd|r\$|\$)?\s*-?[0-9][0-9.,]*\s*[kmb]?$")   # texto solto não vira número
    val = (F.when(~formato_ok, F.lit(None).cast("decimal(18,2)"))
            .when(abreviado.between(-1e16, 1e16), abreviado.cast("decimal(18,2)"))
            .when(cientifica.between(-1e16, 1e16), cientifica.cast("decimal(18,2)"))
            .when(s.rlike(r"^-?\d{1,16}$"), s.cast("decimal(18,2)")))
    return F.when(val > 0, val)   # zerado ou negativo = ausente

def parse_data(df, coluna, saida):
    """Converte datas em múltiplos formatos. try_to_timestamp devolve NULL (sem erro) quando o formato não casa.
       Só vira NULL se NENHUM formato funcionar. Ordem dia/mês antes de mês/dia: '25/12/2020' só casa em d/M/yyyy."""
    df = df.withColumn("_d", F.trim(F.col(coluna)))
    df = df.withColumn("_d", F.regexp_replace("_d", r"(\d)T(\d)", "$1 $2"))                    # ISO com 'T'
    df = df.withColumn("_d", F.regexp_replace("_d", r"(\d{2}:\d{2}:\d{2})(\.\d+)?Z?$", "$1"))  # tira fração/Z
    # Padrões REAIS da origem (verificado nos dados): ISO "yyyy-MM-dd"; barra = "dd/MM/yyyy" (dia > 12 em 6.201 linhas,
    # nunca mês > 12); hífen = "MM-dd-yyyy" (dia > 12 em 2.770 linhas, nunca no 1º campo).
    # Por isso, d-M-yyyy / M/d/yyyy ficam só como último recurso, DEPOIS dos padrões reais.
    formatos = ["yyyy-M-d", "yyyy-M-d HH:mm:ss", "yyyy-M-d HH:mm", "yyyy/M/d", "d/M/yyyy", "M-d-yyyy",
                "yyyyMMdd", "d MMM yyyy", "d MMMM yyyy", "MMM d, yyyy", "MMMM d, yyyy", "MMM d yyyy", "MMMM d yyyy",
                "M/d/yyyy", "d-M-yyyy", "d.M.yyyy", "d/M/yy", "yyyy"]
    tentativas = [F.expr(f"try_to_timestamp(`_d`, '{f}')").cast("date") for f in formatos]
    return df.withColumn(saida, F.coalesce(*tentativas)).drop("_d")

## 7. `silver.tb_cotacao_dolar` (executada primeiro: a taxa é usada no financeiro)

In [ ]:
cot = spark.table("bronze.tb_cotacao_dolar")
# A API devolve várias cotações por dia (boletins intradiários): fica a última do dia
# e, em caso de reingestão do mesmo dia, a ingestão mais recente.
w = Window.partitionBy("data").orderBy(F.col("dataHoraCotacao").desc(), F.col("ingestion_datetime").desc())
cot = (cot.select(F.to_date(F.substring("dataHoraCotacao", 1, 10), "yyyy-MM-dd").alias("data"),
                  to_double("cotacaoCompra").alias("cotacao_compra"),
                  "dataHoraCotacao", "ingestion_datetime")
          .filter("data IS NOT NULL")
          .withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn", "dataHoraCotacao", "ingestion_datetime"))

lim = cot.agg(F.min("data").alias("ini"), F.max("data").alias("fim")).first()
fim = max(lim.fim, date.today())   # série vai até hoje, mesmo que hoje seja fim de semana/feriado

# Série temporal contínua (1 linha por dia corrido)
calendario = spark.sql(f"SELECT explode(sequence(to_date('{lim.ini}'), to_date('{fim}'), interval 1 day)) AS data_cotacao")

# Forward fill: dias sem cotação herdam o último valor disponível
w_ff = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, 0)
df_cotacao = (calendario.join(cot, calendario.data_cotacao == cot.data, "left")
              .withColumn("cotacao_preenchida", F.col("cotacao_compra").isNull())
              .withColumn("cotacao_compra", F.last("cotacao_compra", ignorenulls=True).over(w_ff))
              .select("data_cotacao", "cotacao_compra", "cotacao_preenchida"))
salva_silver(df_cotacao, "silver.tb_cotacao_dolar")
display(spark.table("silver.tb_cotacao_dolar").orderBy(F.desc("data_cotacao")))

## 1. `silver.tb_info_filmes`

In [ ]:
info = spark.table("bronze.tb_movies_info").withColumn("id", F.trim("id"))

# --- Status: normaliza (minúsculas, remove hífens/ruídos/espaços) ANTES de traduzir ---
# Na origem: Released/RELEASED/released, "In-Production", "Post-Production" e até datas caídas na coluna (column shift)
chave_status = F.regexp_replace(F.lower(F.trim("status")), r"[^a-z]", "")   # "Post-Production " -> "postproduction"
status_pt = (F.when(chave_status == "released", "Lançado")
              .when(chave_status == "postproduction", "PósProdução")
              .when(chave_status == "inproduction", "Em Produção")
              .when(chave_status == "planned", "Planejado")
              .when(chave_status == "rumored", "Rumores")
              .when(chave_status.isin("canceled", "cancelled"), "Cancelado")
              .otherwise("Não Informado"))                                    # corrompidos/não mapeáveis

info = parse_data(info, "release_date", "data_lancamento")                    # multi-formato

# Título com caixa corrompida na origem (ex.: "AVENGERS: INFINITY WAR", "spider-man: no way home"):
# se o título está todo em maiúsculas/minúsculas e o título original (mesmo texto, caixa diferente) está
# bem formatado, usa-se a grafia do título original. Nos demais casos o título é mantido como veio.
titulo_limpo = limpa_nulos("title")
orig_limpo = limpa_nulos("original_title")
caixa_ruim = (titulo_limpo == F.upper(titulo_limpo)) | (titulo_limpo == F.lower(titulo_limpo))
titulo_ok = F.when(caixa_ruim & (F.lower(titulo_limpo) == F.lower(orig_limpo)) & (orig_limpo != titulo_limpo),
                   orig_limpo).otherwise(titulo_limpo)

df_info = info.select(
    F.col("id").alias("id_filme"),
    titulo_ok.alias("titulo"),
    orig_limpo.alias("titulo_original"),
    "data_lancamento",
    # duração <= 0 não é válida -> NULL (a origem tem textos como "aa" -> NULL pelo cast seguro)
    F.when(to_int("runtime") > 0, to_int("runtime")).alias("duracao_minutos"),
    limpa_nulos("original_language").alias("idioma_original"),
    status_pt.alias("status_filme"),
    limpa_nulos("overview").alias("sinopse"),
    limpa_nulos("tagline").alias("frase_divulgacao"),
    "ingestion_datetime",
)
# Deduplicação: 1 registro por filme (a origem repete ~5,6 mil filmes, com variações de caixa/formato de data)
df_info = (mais_recente(df_info, ["id_filme"],
                        ["titulo", "data_lancamento", "duracao_minutos", "idioma_original", "sinopse", "frase_divulgacao"])
           .drop("ingestion_datetime")
           .withColumn("ano_lancamento", F.year("data_lancamento")))
salva_silver(df_info, "silver.tb_info_filmes")
display(spark.table("silver.tb_info_filmes").groupBy("status_filme").count())
print("Datas não convertidas (NULL):", spark.table("silver.tb_info_filmes").filter("data_lancamento IS NULL").count())

## 2. `silver.tb_financeiro_filmes`

In [ ]:
# Taxa de cotação: última cotação disponível na silver.tb_cotacao_dolar
taxa = (spark.table("silver.tb_cotacao_dolar").filter("cotacao_compra IS NOT NULL")
        .orderBy(F.desc("data_cotacao")).first().cotacao_compra)
print("Taxa USD->BRL aplicada:", taxa)

fin = (spark.table("bronze.tb_movies_financials")
       .select(F.trim("id").alias("id_filme"),
               limpa_moeda("budget").alias("orcamento_usd"),
               limpa_moeda("revenue").alias("receita_usd"),
               "ingestion_datetime"))
# Duplicatas na origem divergem (ex.: "1.3M" vs 1250000, "Unknown" vs 0): fica a mais recente e, no empate, a mais completa
fin = mais_recente(fin, ["id_filme"], ["orcamento_usd", "receita_usd"]).drop("ingestion_datetime")

def brl(c): return F.round(F.col(c) * F.lit(taxa), 2).cast("decimal(18,2)")

def lucro(rec, orc):
    # Valor ausente NÃO invalida a conta (tratado como 0); só fica NULL se ambos forem ausentes
    return (F.when(F.col(rec).isNull() & F.col(orc).isNull(), F.lit(None))
             .otherwise(F.coalesce(F.col(rec), F.lit(0)) - F.coalesce(F.col(orc), F.lit(0)))
             .cast("decimal(18,2)"))

df_fin = (fin.withColumn("orcamento_brl", brl("orcamento_usd"))
             .withColumn("receita_brl", brl("receita_usd"))
             .withColumn("lucro_usd", lucro("receita_usd", "orcamento_usd"))
             .withColumn("lucro_brl", lucro("receita_brl", "orcamento_brl"))
             # Margem % = lucro / receita * 100; sem divisão por zero (receita <= 0 ou NULL => NULL)
             .withColumn("margem_lucro_pct",
                         F.when(F.col("receita_usd") > 0,
                                F.round(F.col("lucro_usd") / F.col("receita_usd") * 100, 2)).cast("double")))
salva_silver(df_fin, "silver.tb_financeiro_filmes")
display(spark.table("silver.tb_financeiro_filmes").limit(20))

## 3. `silver.tb_metricas_engajamento`

In [ ]:
met = spark.table("bronze.tb_movies_metrics").withColumn("id", F.trim("id"))

def nota(c):
    # Column Shift: texto vira NULL. Fora de 0..10 (inclui escala errada, ex.: 75 em vez de 7.5) vira NULL.
    n = to_double(F.regexp_replace(F.trim(F.col(c)), ",", "."))
    return F.when(n.between(0, 10), n)

popularidade = limpa_decimal("popularity")

df_met = met.select(
    F.col("id").alias("id_filme"),
    F.when(popularidade >= 0, popularidade).alias("popularidade"),   # negativa => NULL
    nota("vote_average").alias("nota_media_tmdb"),
    to_int("vote_count").alias("qtd_votos_tmdb"),                    # negativa/texto => NULL
    nota("averageRating").alias("nota_media_imdb"),
    to_int("numVotes").alias("qtd_votos_imdb"),
    "ingestion_datetime",
)
df_met = mais_recente(df_met, ["id_filme"],
                      ["popularidade", "nota_media_tmdb", "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb"]).drop("ingestion_datetime")
salva_silver(df_met, "silver.tb_metricas_engajamento")
display(spark.table("silver.tb_metricas_engajamento").summary())

## 4. `silver.tb_avaliacoes_usuarios`

In [ ]:
rev = spark.table("bronze.tb_movies_reviews")

sem_texto = F.regexp_replace(F.coalesce(F.col("comentario"), F.lit("")), r"[\s\u00A0]", "") == ""
rev = rev.select(
    F.trim("id").alias("id_filme"),
    F.trim("nome").alias("nome_usuario"),
    F.trim("nota").alias("nota_usuario"),
    F.when(sem_texto, F.lit("Sem comentário")).otherwise(F.trim("comentario")).alias("comentario_usuario"),  # vazio/espaços
)
# Remove duplicatas integrais (filme + usuário + nota + comentário)
rev = rev.dropDuplicates(["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"])
# Nota válida apenas entre 0 e 10; fora da faixa ou texto => NULL
n = to_double("nota_usuario")
df_rev = rev.withColumn("nota_usuario", F.when(n.between(0, 10), n))
salva_silver(df_rev, "silver.tb_avaliacoes_usuarios")

## 5. `silver.tb_generos` e 6. `silver.tb_pessoas_empresas`

In [ ]:
COLS_LISTA = ["genres", "cast", "directors", "writers", "production_companies"]
cred = spark.table("bronze.tb_credits_and_tags")
# "[]", "N/A", "Nenhum", "" etc. = ausência de dado
cred = cred.select(F.trim("id").alias("id"), *[limpa_nulos(c).alias(c) for c in COLS_LISTA], "ingestion_datetime")
cred = mais_recente(cred, ["id"], COLS_LISTA).drop("ingestion_datetime")   # 1 linha por filme (a origem repete filmes)

# Domínio de gêneros (catálogo TMDB). Tudo que não pertence a ele é resíduo de column shift.
GENEROS_VALIDOS = ["Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary", "Drama", "Family", "Fantasy",
                   "History", "Horror", "Music", "Mystery", "Romance", "Science Fiction", "TV Movie", "Thriller", "War", "Western"]
GENEROS_LOWER = [g.lower() for g in GENEROS_VALIDOS]
# Idiomas que "vazam" para colunas de pessoas/empresas na origem
IDIOMAS_BASE = ["english", "french", "spanish", "german", "italian", "japanese", "korean", "mandarin", "cantonese", "chinese",
           "portuguese", "russian", "hindi", "arabic", "turkish", "no language", "latin"]

# Países/idiomas que "vazam" para diretores/roteiristas/produtoras: o próprio dataset diz quais são
# (valores frequentes das colunas production_countries e spoken_languages) + termos soltos vistos na origem.
def tokens_distintos(coluna):
    s = F.regexp_replace(F.col(coluna), r'[\[\]"\\]', "")
    s = F.regexp_replace(s, r"[;|]", ",")
    t = (spark.table("bronze.tb_credits_and_tags").select(F.explode(F.split(s, ",")).alias("t"))
         .select(F.lower(F.trim("t")).alias("t")).filter("t != ''")
         .groupBy("t").count().filter("count >= 30"))      # países/idiomas são frequentes; lixo deslocado é raro
    return [r.t for r in t.collect()]

# Só vira exclusão o que é curto (<= 40 chars): países/idiomas são nomes curtos; evita puxar fragmentos de texto
PAISES_IDIOMAS = [t for t in tokens_distintos("production_countries") + tokens_distintos("spoken_languages") if len(t) <= 40]
IDIOMAS = IDIOMAS_BASE + PAISES_IDIOMAS + ["short film", "short", "documentary film"]

def explode_lista(df, coluna, saida):
    # Padroniza separadores (',' ';' '|'), explode e remove resíduos de column shift.
    s = F.regexp_replace(F.col(coluna), r'[\[\]"\\]', "")           # restos de aspas/colchetes/barras
    s = F.regexp_replace(s, r"[;|]", ",")                            # unifica separadores (vírgula, ponto e vírgula, pipe)
    out = df.select(F.col("id").alias("id_filme"), F.explode(F.split(s, ",")).alias(saida))
    out = out.withColumn(saida, F.regexp_replace(F.trim(F.regexp_replace(F.col(saida), r"\s+", " ")), r"^'+|'+$", ""))
    return out.filter(~F.lower(F.col(saida)).isin(NULOS))             # resíduos em branco/ausência de dado

# --- Gêneros: só valores do domínio ---
g = explode_lista(cred, "genres", "genero")
g = (g.withColumn("genero", F.initcap(F.lower("genero")))
      .withColumn("genero", F.when(F.col("genero") == "Tv Movie", "TV Movie").otherwise(F.col("genero")))
      .filter(F.col("genero").isin(GENEROS_VALIDOS))
      .dropDuplicates(["id_filme", "genero"]))
salva_silver(g, "silver.tb_generos")
display(spark.table("silver.tb_generos").groupBy("genero").count().orderBy(F.desc("count")))

# --- Pessoas e empresas: 4 colunas -> 1 dimensão unificada ---
def residuo_de_shift(c, eh_pessoa):
    palavras = F.size(F.split(c, " "))
    r = (F.length(c) < 2) | (F.length(c) > 100)                          # vazio/curto demais ou texto descritivo longo
    r = r | c.rlike(r"^[0-9.\s\-]+$")                                    # número deslocado (ex.: popularidade 0.6)
    r = r | (c.rlike(r"^[a-zà-ÿ]") & (palavras >= 4))                    # fragmento de frase (começa em minúscula)
    r = r | F.lower(c).isin(GENEROS_LOWER + IDIOMAS)                     # gênero/idioma caído na coluna errada
    if eh_pessoa:
        r = r | c.rlike(r"\d")                                           # nome de pessoa não tem dígito
    return r

TIPOS = {"cast": "Ator", "directors": "Diretor", "writers": "Roteirista", "production_companies": "Produtora"}
partes = []
for coluna, tipo in TIPOS.items():
    p = explode_lista(cred, coluna, "nome")
    p = p.filter(~residuo_de_shift(F.col("nome"), tipo != "Produtora")).withColumn("tipo_entidade", F.lit(tipo))
    partes.append(p)
pe = partes[0]
for p in partes[1:]:
    pe = pe.unionByName(p)
pe = pe.withColumn("nome", F.initcap(F.lower("nome")))        # capitalização padronizada
pe = pe.dropDuplicates(["id_filme", "nome", "tipo_entidade"])
salva_silver(pe, "silver.tb_pessoas_empresas")
display(spark.table("silver.tb_pessoas_empresas").groupBy("tipo_entidade").count())